In [0]:
%pip install requests

import requests

response = requests.get(
    f"https://api.weatherapi.com/v1/current.json?key={"87dd93c68bf142bc92b93416251909"}&q={"Warangal"}"
)
print(response.json())

In [0]:
# Import the requests library to make HTTP requests to the weather API
import requests
# Import time module to handle timing and sleep intervals
import time
# Import Spark SQL types to define the schema for the DataFrame
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
# Import Row to create row objects for the DataFrame
from pyspark.sql import Row
# Import datetime to capture the current UTC timestamp
from datetime import datetime

# Set your API key for authentication with the weather API
api_key = "87dd93c68bf142bc92b93416251909"
# Specify the location for which to fetch weather data
location = "Warangal"  # Change as needed
# Construct the API URL with the provided API key and location
url = f"https://api.weatherapi.com/v1/current.json?key={"87dd93c68bf142bc92b93416251909"}&q={"Warangal"}"

# Define the schema for the Spark DataFrame to ensure correct data types
schema = StructType([
    StructField('name', StringType(), True),
    StructField('region', StringType(), True),
    StructField('country', StringType(), True),
    StructField("temp_c", DoubleType(), True),
    StructField('humidity', DoubleType(), True),
    StructField("condition", StringType(), True),
    StructField("timestamp", TimestampType(), True)
])

# Initialize an empty list to store weather data rows
data = []
# Record the start time to control the duration of data collection
start_time = time.time()
# Loop for 120 seconds, fetching weather data every 5 seconds
while time.time() - start_time < 120:
    # Make a GET request to the weather API
    response = requests.get(url)
    # If the request is successful, process the response
    if response.status_code == 200:
        weather = response.json()
        # Create a Row object with the relevant weather data and current timestamp
        row = Row(
            name=weather["location"]["name"],
            region=weather["location"]["region"],
            country=weather["location"]["country"],
            temp_c=weather["current"]["temp_c"],
            humidity=weather["current"]["humidity"],
            condition=weather["current"]["condition"]["text"],
            timestamp=datetime.utcnow()
        )
        # Append the row to the data list
        data.append(row)
    # Wait for 5 seconds before making the next API call to avoid rate limits
    time.sleep(5)  # Adjust frequency as needed

# Create a Spark DataFrame from the collected data using the defined schema
df = spark.createDataFrame(data, schema)
# Write the DataFrame to a Delta table in append mode for persistent storage
df.write.mode("append").saveAsTable("my_catalog1.weather_db.weather_data")

In [0]:
type(data[0])